# 🛡️ NetGuard AI — Hybrid Attention-Based Intrusion Detection System
## Real-Time Intrusion Detection and Alerting System for Software-Defined Networks

**Model:** Hybrid Attention-Based Deep Neural Network (GradientBoosting Attention + MLP Classifier)

---

## 1. Setup & Installation

In [ ]:
!pip install -q scikit-learn imbalanced-learn matplotlib seaborn pandas numpy

## 2. Mount Google Drive & Load Dataset
Upload your CICIDS 2017 CSV files to a folder called **CICIDS2017** in your Google Drive, then run this cell.

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to your CSV files in Google Drive
# Change this path if your folder is named differently
DATASET_PATH = '/content/drive/MyDrive/CICIDS2017'

csv_files = sorted(glob.glob(os.path.join(DATASET_PATH, '*.csv')))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'  📄 {os.path.basename(f)} ({size_mb:.1f} MB)')

## 3. Load & Explore Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load all CSVs
dfs = []
for f in csv_files:
    print(f'Loading: {os.path.basename(f)}')
    df = pd.read_csv(f, low_memory=False)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.columns = data.columns.str.strip()

print(f'\n📊 Dataset Shape: {data.shape}')
print(f'   Rows: {data.shape[0]:,}')
print(f'   Features: {data.shape[1]}')
print(f'\nColumn Names:\n{list(data.columns)}')

In [ ]:
# Dataset info
print('=== Dataset Info ===')
print(f'Total samples: {len(data):,}')
print(f'Total features: {data.shape[1]}')
print(f'\nMissing values: {data.isnull().sum().sum()}')
print(f'Infinite values: {np.isinf(data.select_dtypes(include=[np.number])).sum().sum()}')
print(f'\nData types:\n{data.dtypes.value_counts()}')
print(f'\n=== First 5 Rows ===')
data.head()

## 4. Data Visualization — Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Class distribution
label_counts = data['Label'].value_counts()
print('=== Label Distribution ===')
for label, count in label_counts.items():
    pct = count / len(data) * 100
    print(f'  {label}: {count:>8,} ({pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart
colors = sns.color_palette('husl', len(label_counts))
bars = axes[0].barh(label_counts.index, label_counts.values, color=colors)
axes[0].set_xlabel('Number of Samples')
axes[0].set_title('Traffic Class Distribution', fontweight='bold', fontsize=14)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)

# Pie chart (excluding BENIGN for better view of attacks)
attack_counts = label_counts[label_counts.index != 'BENIGN']
axes[1].pie(attack_counts.values, labels=attack_counts.index, autopct='%1.1f%%', colors=sns.color_palette('Set2', len(attack_counts)))
axes[1].set_title('Attack Type Distribution (Excluding Benign)', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

## 5. Data Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('=== Preprocessing Pipeline ===')

# Step 1: Clean data
print('\n1. Cleaning data...')
data.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(data)
data.dropna(inplace=True)
print(f'   Removed {before - len(data):,} rows with inf/NaN values')
print(f'   Remaining: {len(data):,} rows')

# Step 2: Sample for faster training (use 50,000 rows)
SAMPLE_SIZE = 50000
if len(data) > SAMPLE_SIZE:
    data = data.sample(n=SAMPLE_SIZE, random_state=42)
    print(f'\n2. Sampled to {SAMPLE_SIZE:,} rows for training efficiency')

# Step 3: Separate features and labels
print('\n3. Separating features and labels...')
labels = data['Label'].copy()
features = data.drop(columns=['Label'])
features = features.select_dtypes(include=[np.number])
print(f'   Features: {features.shape[1]} numeric columns')
print(f'   Feature names: {list(features.columns)[:10]}... (showing first 10)')

# Step 4: Encode labels
print('\n4. Encoding labels...')
le = LabelEncoder()
y = le.fit_transform(labels)
print(f'   Classes ({len(le.classes_)}): {list(le.classes_)}')

# Step 5: Scale features
print('\n5. Scaling features...')
scaler = StandardScaler()
X = scaler.fit_transform(features.values.astype(np.float32))
print(f'   Final shape: X={X.shape}, y={y.shape}')
print('\n✅ Preprocessing complete!')

## 6. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Testing set:  {X_test.shape[0]:,} samples')
print(f'\nTrain label distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {le.classes_[u]}: {c:,}')

## 7. SMOTE Oversampling (Balancing Minority Classes)
We apply **SMOTE** to balance rare attack types like Bot, Heartbleed, and Infiltration that have very few samples.

In [ ]:
from imblearn.over_sampling import SMOTE

print('=== SMOTE Oversampling ===')
unique_train, counts_train = np.unique(y_train, return_counts=True)
min_count = counts_train.min()
print(f'Before SMOTE: {len(X_train):,} samples')
print(f'Min class count: {min_count}')

# Cap each class at max 3000 samples
max_per_class = 3000
sampling_strategy = {}
for cls, cnt in zip(unique_train, counts_train):
    sampling_strategy[cls] = max(cnt, min(max_per_class, cnt * 10))

k = min(5, min_count - 1) if min_count > 1 else 1
smote = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=sampling_strategy)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE:  {len(X_train_sm):,} samples')
print(f'\nClass distribution after SMOTE:')
unique_after, counts_after = np.unique(y_train_sm, return_counts=True)
for u, c in zip(unique_after, counts_after):
    print(f'  {le.classes_[u]}: {c:,}')

# Visualize before vs after
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar([le.classes_[u] for u in unique_train], counts_train, color='steelblue')
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar([le.classes_[u] for u in unique_after], counts_after, color='coral')
axes[1].set_title('After SMOTE', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8. Hybrid Attention-Based Model Architecture

Our model has two stages:
1. **Attention Mechanism** — GradientBoosting learns which features are most important
2. **Deep Neural Network** — MLP classifies traffic using attention-weighted features

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
import time

# Stage 1: Compute Attention Weights using GradientBoosting
print('=== Stage 1: Computing Attention Weights ===')
print('Training GradientBoosting for feature importance...')
start = time.time()

gb = GradientBoostingClassifier(
    n_estimators=50, max_depth=4, random_state=42,
    subsample=0.8, learning_rate=0.1
)
gb.fit(X_train, y_train)  # Fit on original data (before SMOTE) for speed

attention_weights = gb.feature_importances_
attention_weights = attention_weights / (attention_weights.max() + 1e-8)

gb_time = time.time() - start
print(f'Attention computed in {gb_time:.1f}s')
print(f'\nTop 10 most important features:')
top_indices = np.argsort(attention_weights)[-10:][::-1]
feature_names = list(features.columns)
for i in top_indices:
    print(f'  {feature_names[i]}: {attention_weights[i]:.4f}')

In [ ]:
# Visualize attention weights
plt.figure(figsize=(14, 6))
sorted_idx = np.argsort(attention_weights)[-20:]
plt.barh([feature_names[i] for i in sorted_idx], attention_weights[sorted_idx], color='#3b82f6')
plt.xlabel('Attention Weight')
plt.title('Top 20 Feature Attention Weights', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Stage 2: Train MLP on Attention-Weighted Features
print('=== Stage 2: Training MLP Classifier ===')
X_train_weighted = X_train_sm * attention_weights
X_test_weighted = X_test * attention_weights

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.01,
    max_iter=15,
    batch_size=256,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=True,
)

start = time.time()
mlp.fit(X_train_weighted, y_train_sm)
mlp_time = time.time() - start
print(f'\n✅ MLP trained in {mlp_time:.1f}s')
print(f'Total training time: {gb_time + mlp_time:.1f}s')

## 9. Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Predict on test set
y_pred = mlp.predict(X_test_weighted)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('╔══════════════════════════════════════════╗')
print('║     MODEL PERFORMANCE SUMMARY            ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Accuracy:  {accuracy*100:>7.2f}%                   ║')
print(f'║  Precision: {precision*100:>7.2f}%                   ║')
print(f'║  Recall:    {recall*100:>7.2f}%                   ║')
print(f'║  F1-Score:  {f1*100:>7.2f}%                   ║')
print('╚══════════════════════════════════════════╝')

In [ ]:
# Detailed Classification Report
print('=== Detailed Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, linecolor='gray')
plt.xlabel('Predicted Label', fontsize=13)
plt.ylabel('True Label', fontsize=13)
plt.title('Confusion Matrix — Hybrid Attention Model', fontweight='bold', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Per-Class Performance Analysis

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1s, sup = precision_recall_fscore_support(y_test, y_pred, zero_division=0)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x = np.arange(len(le.classes_))
axes[0].barh(x, prec, color='#3b82f6')
axes[0].set_yticks(x); axes[0].set_yticklabels(le.classes_)
axes[0].set_xlabel('Score'); axes[0].set_title('Precision per Class', fontweight='bold')
axes[0].set_xlim(0, 1.1)
for i, v in enumerate(prec): axes[0].text(v + 0.01, i, f'{v:.2f}', va='center')

axes[1].barh(x, rec, color='#10b981')
axes[1].set_yticks(x); axes[1].set_yticklabels(le.classes_)
axes[1].set_xlabel('Score'); axes[1].set_title('Recall per Class', fontweight='bold')
axes[1].set_xlim(0, 1.1)
for i, v in enumerate(rec): axes[1].text(v + 0.01, i, f'{v:.2f}', va='center')

axes[2].barh(x, f1s, color='#f59e0b')
axes[2].set_yticks(x); axes[2].set_yticklabels(le.classes_)
axes[2].set_xlabel('Score'); axes[2].set_title('F1-Score per Class', fontweight='bold')
axes[2].set_xlim(0, 1.1)
for i, v in enumerate(f1s): axes[2].text(v + 0.01, i, f'{v:.2f}', va='center')

plt.tight_layout()
plt.show()

## 12. Training Loss Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mlp.loss_curve_, 'b-', linewidth=2, label='Training Loss')
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_:
    plt.plot(mlp.validation_scores_, 'r--', linewidth=2, label='Validation Score')
plt.xlabel('Iteration')
plt.ylabel('Loss / Score')
plt.title('Training Progress', fontweight='bold', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Feature Correlation Heatmap

In [ ]:
# Top 15 features correlation
top15_idx = np.argsort(attention_weights)[-15:][::-1]
top15_names = [feature_names[i] for i in top15_idx]
corr = features[top15_names].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation (Top 15 Attention Features)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 14. Sample Predictions

In [ ]:
# Show sample predictions with confidence
proba = mlp.predict_proba(X_test_weighted[:20])
preds = mlp.predict(X_test_weighted[:20])

print('=== Sample Predictions ===')
print(f'{"#":<4} {"Actual":<25} {"Predicted":<25} {"Confidence":<12} {"Correct"}')
print('-' * 80)
for i in range(20):
    actual = le.classes_[y_test[i]]
    predicted = le.classes_[preds[i]]
    conf = proba[i].max() * 100
    correct = '✅' if actual == predicted else '❌'
    print(f'{i+1:<4} {actual:<25} {predicted:<25} {conf:>8.1f}%    {correct}')

## 15. Summary & Architecture Diagram

In [ ]:
print('╔══════════════════════════════════════════════════════════════╗')
print('║          NetGuard AI — Model Summary                       ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Model:        Hybrid Attention + MLP                      ║')
print(f'║  Attention:    GradientBoosting (50 trees, depth 4)        ║')
print(f'║  Classifier:   MLP (64 → 32 → {len(le.classes_)} classes)              ║')
print(f'║  SMOTE:        Applied (max 3,000 per class)               ║')
print(f'║  Features:     {X.shape[1]} numeric network flow features          ║')
print(f'║  Classes:      {len(le.classes_)} traffic types                         ║')
print(f'║  Accuracy:     {accuracy*100:.2f}%                                   ║')
print(f'║  Training:     {gb_time + mlp_time:.1f}s total                              ║')
print('╚══════════════════════════════════════════════════════════════╝')
print()
print('Architecture:')
print('  ┌─────────────────┐')
print('  │  Network Flows  │  (78 features per flow)')
print('  └────────┬────────┘')
print('           ▼')
print('  ┌─────────────────┐')
print('  │  Preprocessing  │  Clean → Scale → Encode')
print('  └────────┬────────┘')
print('           ▼')
print('  ┌─────────────────┐')
print('  │ SMOTE Balancing │  Oversample minority attacks')
print('  └────────┬────────┘')
print('           ▼')
print('  ┌─────────────────┐')
print('  │   Attention     │  GradientBoosting feature weights')
print('  │   Mechanism     │  (identifies important features)')
print('  └────────┬────────┘')
print('           ▼')
print('  ┌─────────────────┐')
print('  │  MLP Classifier │  64 → 32 → Output')
print('  │  (Deep NN)      │  (classifies attack types)')
print('  └────────┬────────┘')
print('           ▼')
print('  ┌─────────────────┐')
print(f'  │   {len(le.classes_)} Classes     │  BENIGN + {len(le.classes_)-1} Attack Types')
print('  └─────────────────┘')